# 🔬 GenAI Energy-Quality Benchmark (Universal)
**Works on T4 / A100 / H100** — auto-detects GPU

Phase 1: Energy measurement (all models)
Phase 2: Quality measurement (CLIP, perplexity)

Cell 1 → 런타임 다시 시작 → Cell 2부터 순서대로!

In [ ]:
# Cell 1: Install dependencies → RESTART RUNTIME after this
!pip install -q diffusers accelerate pynvml scipy open_clip_torch
# Clear cached custom model code
import shutil, os
cache_path = os.path.expanduser('~/.cache/huggingface/modules/transformers_modules')
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
import transformers
print(f'transformers: {transformers.__version__}')
print('✅ Restart runtime now, then run Cell 2!')

In [ ]:
# Cell 2: Setup
import torch, time, json, os, gc, datetime
import numpy as np
import pynvml
from threading import Thread, Event

assert torch.cuda.is_available(), 'No GPU!'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/project_energy_v2'
except:
    SAVE_DIR = '/content/project_energy_v2'
os.makedirs(SAVE_DIR, exist_ok=True)

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
GPU_NAME = pynvml.nvmlDeviceGetName(handle)
if isinstance(GPU_NAME, bytes): GPU_NAME = GPU_NAME.decode()
gpu_tag = GPU_NAME.replace(' ','-').replace('/','-')
BACKEND = 'cuda'
REPEATS = 30

print(f'GPU: {GPU_NAME}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')
print(f'Save: {SAVE_DIR}')

class PowerMonitor:
    def __init__(self):
        self.samples = []
        self._stop = Event()
    def _record(self):
        while not self._stop.is_set():
            try:
                p = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
                self.samples.append((time.time(), p))
            except: pass
            time.sleep(0.1)
    def start(self):
        self.samples = []
        self._stop.clear()
        self._thread = Thread(target=self._record, daemon=True)
        self._thread.start()
    def stop(self):
        self._stop.set()
        self._thread.join()
        if len(self.samples) < 2: return {}
        times, powers = zip(*self.samples)
        dt = [times[i+1]-times[i] for i in range(len(times)-1)]
        energy = sum((powers[i]+powers[i+1])/2*dt[i] for i in range(len(dt)))
        return {'total_energy_j': round(energy,2), 'avg_power_w': round(np.mean(powers),2),
                'max_power_w': round(max(powers),2), 'duration_sec': round(times[-1]-times[0],2),
                'samples': len(self.samples)}

def save(data, name):
    path = f'{SAVE_DIR}/{name}_{gpu_tag}.json'
    with open(path, 'w') as f: json.dump(data, f, indent=2)
    print(f'  Saved {len(data)} records to {path}')

print('✅ Ready')

In [ ]:
# Cell 3: TEXT — Phi-3 (3.8B)
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = 'microsoft/Phi-3-mini-4k-instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda')
prompt = 'Write a short story about a robot learning to paint.'
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

results = []
for max_tok in [32, 64, 128, 192, 256, 384, 512]:
    print(f'  Phi-3 {max_tok} tokens')
    _ = model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)
    for run in range(1, REPEATS+1):
        torch.cuda.synchronize()
        pm = PowerMonitor(); pm.start()
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)
        torch.cuda.synchronize()
        gen_time = time.time() - t0
        # Get generated text for quality
        gen_text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        stats = pm.stop()
        stats.update({'generation_time_sec': round(gen_time,2), 'max_tokens': max_tok,
                      'actual_tokens': len(out[0]) - inputs['input_ids'].shape[1],
                      'modality': 'text', 'model': 'Phi-3-mini-4k', 'params_B': 3.8,
                      'run': run, 'hardware': GPU_NAME, 'backend': BACKEND})
        if run == 1: stats['sample_text'] = gen_text[:200]
        results.append(stats)

save(results, 'text_phi3')
del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
print(f'✅ Phi-3 done: {len(results)} runs')

In [ ]:
# Cell 4: TEXT — Gemma-2-2B
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = 'google/gemma-2-2b-it'
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda')
prompt = 'Write a short story about a robot learning to paint.'
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

results = []
for max_tok in [32, 64, 128, 256, 512]:
    print(f'  Gemma-2 {max_tok} tokens')
    _ = model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)
    for run in range(1, REPEATS+1):
        torch.cuda.synchronize()
        pm = PowerMonitor(); pm.start()
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=max_tok, do_sample=False)
        torch.cuda.synchronize()
        stats = pm.stop()
        stats.update({'generation_time_sec': round(time.time()-t0,2), 'max_tokens': max_tok,
                      'actual_tokens': len(out[0]) - inputs['input_ids'].shape[1],
                      'modality': 'text', 'model': 'Gemma-2-2B', 'params_B': 2.0,
                      'run': run, 'hardware': GPU_NAME, 'backend': BACKEND})
        results.append(stats)

save(results, 'text_gemma2')
del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
print(f'✅ Gemma-2 done: {len(results)} runs')

In [ ]:
# Cell 5: IMAGE — SD v1.5 (energy + CLIP quality)
from diffusers import StableDiffusionPipeline
import open_clip
from PIL import Image
import torchvision.transforms as T

# Load CLIP for quality measurement
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_model = clip_model.to('cuda').eval()
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')

def get_clip_score(image, text):
    img_input = clip_preprocess(image).unsqueeze(0).to('cuda')
    txt_input = clip_tokenizer([text]).to('cuda')
    with torch.no_grad():
        img_feat = clip_model.encode_image(img_input)
        txt_feat = clip_model.encode_text(txt_input)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
        score = (img_feat @ txt_feat.T).item()
    return round(score, 4)

pipe = StableDiffusionPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16).to('cuda')
prompt = 'A serene landscape with mountains and a lake at sunset, highly detailed, 8k'

results = []
for res in [128, 192, 256, 320, 384, 448, 512, 640]:
    for steps in [5, 10, 15, 20, 30, 50]:
        print(f'  SD v1.5 {res}x{res} {steps}steps')
        _ = pipe(prompt, height=res, width=res, num_inference_steps=steps,
                guidance_scale=7.5, generator=torch.Generator('cuda').manual_seed(0))
        for run in range(1, REPEATS+1):
            torch.cuda.synchronize()
            pm = PowerMonitor(); pm.start()
            t0 = time.time()
            output = pipe(prompt, height=res, width=res, num_inference_steps=steps,
                    guidance_scale=7.5, generator=torch.Generator('cuda').manual_seed(run))
            torch.cuda.synchronize()
            gen_time = time.time() - t0
            stats = pm.stop()
            # CLIP score
            clip_s = get_clip_score(output.images[0], prompt)
            stats.update({'generation_time_sec': round(gen_time,2), 'resolution': f'{res}x{res}',
                          'steps': steps, 'clip_score': clip_s,
                          'modality': 'image', 'model': 'SD-v1-5', 'params_B': 0.9,
                          'run': run, 'hardware': GPU_NAME, 'backend': BACKEND})
            results.append(stats)
        # Print progress
        avg_clip = np.mean([r['clip_score'] for r in results[-REPEATS:]])
        avg_e = np.mean([r.get('total_energy_j',0) for r in results[-REPEATS:]])
        print(f'    → {avg_e:.0f}J, CLIP={avg_clip:.3f}')

save(results, 'image_sd15')
del pipe; gc.collect(); torch.cuda.empty_cache()
print(f'✅ SD v1.5 done: {len(results)} runs')

In [ ]:
# Cell 6: IMAGE — SDXL (energy + CLIP quality)
from diffusers import StableDiffusionXLPipeline

pipe = StableDiffusionXLPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16, variant='fp16').to('cuda')
prompt = 'A serene landscape with mountains and a lake at sunset, highly detailed, 8k'

results = []
for res in [512, 640, 768, 896, 1024]:
    for steps in [5, 10, 15, 20, 30]:
        print(f'  SDXL {res}x{res} {steps}steps')
        try:
            _ = pipe(prompt, height=res, width=res, num_inference_steps=steps,
                    generator=torch.Generator('cuda').manual_seed(0))
        except: continue
        for run in range(1, REPEATS+1):
            torch.cuda.synchronize()
            pm = PowerMonitor(); pm.start()
            t0 = time.time()
            output = pipe(prompt, height=res, width=res, num_inference_steps=steps,
                    generator=torch.Generator('cuda').manual_seed(run))
            torch.cuda.synchronize()
            gen_time = time.time() - t0
            stats = pm.stop()
            clip_s = get_clip_score(output.images[0], prompt)
            stats.update({'generation_time_sec': round(gen_time,2), 'resolution': f'{res}x{res}',
                          'steps': steps, 'clip_score': clip_s,
                          'modality': 'image', 'model': 'SDXL-base-1.0', 'params_B': 3.5,
                          'run': run, 'hardware': GPU_NAME, 'backend': BACKEND})
            results.append(stats)
        avg_clip = np.mean([r['clip_score'] for r in results[-REPEATS:]])
        avg_e = np.mean([r.get('total_energy_j',0) for r in results[-REPEATS:]])
        print(f'    → {avg_e:.0f}J, CLIP={avg_clip:.3f}')

save(results, 'image_sdxl')
del pipe; gc.collect(); torch.cuda.empty_cache()
print(f'✅ SDXL done: {len(results)} runs')

In [ ]:
# Cell 7: VIDEO — AnimateDiff (energy)
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
adapter = MotionAdapter.from_pretrained('guoyww/animatediff-motion-adapter-v1-5-3',
    torch_dtype=torch.float16)
vpipe = AnimateDiffPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5',
    motion_adapter=adapter, torch_dtype=torch.float16).to('cuda')
vpipe.scheduler = DDIMScheduler.from_config(vpipe.scheduler.config,
    beta_schedule='linear', clip_sample=False, timestep_spacing='linspace', steps_offset=1)
prompt = 'A serene landscape with mountains and a lake at sunset, highly detailed, 8k'

results = []
for frames in [4, 6, 8, 10, 12, 16]:
    for steps in [10, 20, 30]:
        print(f'  AnimateDiff {frames}f {steps}steps')
        try:
            _ = vpipe(prompt, num_frames=frames, height=256, width=256,
                     num_inference_steps=steps, generator=torch.Generator('cuda').manual_seed(0))
        except: continue
        for run in range(1, REPEATS+1):
            torch.cuda.synchronize()
            pm = PowerMonitor(); pm.start()
            t0 = time.time()
            _ = vpipe(prompt, num_frames=frames, height=256, width=256,
                     num_inference_steps=steps, generator=torch.Generator('cuda').manual_seed(run))
            torch.cuda.synchronize()
            stats = pm.stop()
            stats.update({'generation_time_sec': round(time.time()-t0,2), 'frames': frames,
                          'steps': steps, 'resolution': '256x256', 'modality': 'video',
                          'model': 'AnimateDiff-v1.5', 'run': run,
                          'hardware': GPU_NAME, 'backend': BACKEND})
            results.append(stats)

save(results, 'video_animatediff')
del vpipe, adapter; gc.collect(); torch.cuda.empty_cache()
print(f'✅ AnimateDiff done: {len(results)} runs')

In [ ]:
# Cell 8: Install audiocraft (AFTER text experiments)
!apt-get install -y -qq libavformat-dev libavdevice-dev
!pip install -q av audiocraft
print('✅ audiocraft installed')

In [ ]:
# Cell 9: MUSIC — MusicGen
from audiocraft.models import MusicGen

results = []
for model_name, params_m in [('small', 589), ('medium', 2008)]:
    mg = MusicGen.get_pretrained(f'facebook/musicgen-{model_name}')
    desc = ['happy rock song with electric guitar and drums']
    for max_tok in [64, 128, 256, 512]:
        mg.set_generation_params(use_sampling=True, top_k=250, max_tokens=max_tok)
        audio_sec = round(max_tok / 50, 1)
        print(f'  MusicGen-{model_name} {max_tok}tok ({audio_sec}s)')
        _ = mg.generate(desc)
        for run in range(1, REPEATS+1):
            torch.cuda.synchronize()
            pm = PowerMonitor(); pm.start()
            t0 = time.time()
            wav = mg.generate(desc)
            torch.cuda.synchronize()
            stats = pm.stop()
            stats.update({'generation_time_sec': round(time.time()-t0,2), 'audio_sec': audio_sec,
                          'max_tokens': max_tok, 'modality': 'music',
                          'model': f'MusicGen-{model_name}', 'params_M': params_m,
                          'run': run, 'hardware': GPU_NAME, 'backend': BACKEND})
            results.append(stats)
    del mg; gc.collect(); torch.cuda.empty_cache()

save(results, 'music_musicgen')
print(f'✅ MusicGen done: {len(results)} runs')

In [ ]:
# Cell 10: Batched inference (Phi-3)
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = 'microsoft/Phi-3-mini-4k-instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda')
prompt = 'Write a short story about a robot learning to paint.'

results = []
for batch_size in [1, 2, 4, 8, 16]:
    batch_prompts = [prompt] * batch_size
    inputs = tokenizer(batch_prompts, return_tensors='pt', padding=True).to('cuda')
    print(f'  Batch size={batch_size}')
    _ = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    for run in range(1, REPEATS+1):
        torch.cuda.synchronize()
        pm = PowerMonitor(); pm.start()
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        torch.cuda.synchronize()
        gen_time = time.time() - t0
        stats = pm.stop()
        per_q_e = stats.get('total_energy_j', 0) / batch_size
        stats.update({'generation_time_sec': round(gen_time,2), 'max_tokens': 256,
                      'batch_size': batch_size, 'per_query_energy_j': round(per_q_e, 2),
                      'per_query_time_sec': round(gen_time/batch_size, 2),
                      'modality': 'text_batched', 'model': 'Phi-3-mini-4k', 'params_B': 3.8,
                      'run': run, 'hardware': GPU_NAME, 'backend': BACKEND})
        results.append(stats)

save(results, 'batched_phi3')
del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
print(f'✅ Batched done: {len(results)} runs')

In [ ]:
# Cell 11: Summary
print(f'\n🎉 ALL EXPERIMENTS COMPLETE on {GPU_NAME}!')
print(f'Results saved to: {SAVE_DIR}')
# List all saved files
for f in sorted(os.listdir(SAVE_DIR)):
    if f.endswith('.json') and gpu_tag in f:
        path = os.path.join(SAVE_DIR, f)
        with open(path) as fh:
            data = json.load(fh)
        print(f'  {f}: {len(data)} records')